In [139]:
# ============================================================
# 1. ACCESO ARCHIVOS GOOGLE DRIVE
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [140]:
# ============================================================
# 2. IMPORTAR LIBRERÍAS
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

In [141]:
# ============================================================
# 3. DEFINIR RUTAS
  # Ruta del Excel de salarios y la ruta donde quieres guardar el CSV limpio.
# ============================================================

# Ruta del Excel bruto de salarios
salary_file = "/content/drive/MyDrive/TFG-Accesibilidad-Vivienda-Joven/data/data raw/INE_Ganancia Media Mensual por Trabajador (2010-2023).xlsx"

# Ruta de salida del CSV limpio
output_file = "/content/drive/MyDrive/TFG-Accesibilidad-Vivienda-Joven/data/data clean/salary_observed.csv"

print("Archivo de entrada:")
print(salary_file)

print("\nArchivo de salida:")
print(output_file)

Archivo de entrada:
/content/drive/MyDrive/TFG-Accesibilidad-Vivienda-Joven/data/data raw/INE_Ganancia Media Mensual por Trabajador (2010-2023).xlsx

Archivo de salida:
/content/drive/MyDrive/TFG-Accesibilidad-Vivienda-Joven/data/data clean/salary_observed.csv


In [142]:
# ============================================================
# 4. LEER EL EXCEL BRUTO
# ============================================================

df_raw = pd.read_excel(salary_file, sheet_name="tabla-28201", header=None)

print("Dimensiones del Excel bruto:", df_raw.shape)
display(df_raw.head(25))

Dimensiones del Excel bruto: (1654, 4)


,0,1,2,3
0,Resultados nacionales y por comunidades aútonomas,NaN,NaN,NaN
1,Ganancia media anual por trabajador,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN
3,Sexo y edad,NaN,NaN,NaN
4,Unidades: €,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN
6,,Ambos sexos,Mujeres,Hombres
7,Total Nacional,NaN,NaN,NaN
8,Todas las edades,NaN,NaN,NaN
9,2023,28049.94,25591.31,30372.49


In [143]:
# ============================================================
# 5. LOCALIZAR EL INICIO Y EL FINAL DEL BLOQUE DE DATOS
  # Queremos saber:
    # - la fila donde empieza 'Total Nacional'
    # - la fila donde empiezan las 'Notas:'
# ============================================================

start_idx = df_raw[df_raw[0].astype(str).str.strip() == "Total Nacional"].index.min()
end_idx = df_raw[df_raw[0].astype(str).str.contains("Notas:", na=False)].index.min()

print("Fila donde empieza 'Total Nacional':", start_idx)
print("Fila donde empieza 'Notas:':", end_idx)

Fila donde empieza 'Total Nacional': 7
Fila donde empieza 'Notas:': 1647


In [144]:
# ============================================================
# 6. RECORTAR SOLO EL BLOQUE ÚTIL
  # Desde 'Total Nacional' hasta justo antes de 'Notas:'.
# ============================================================

df_block = df_raw.loc[start_idx:end_idx - 1].copy()

# Renombramos columnas útiles
df_block.columns = ["raw_label", "both_sexes", "women", "men"]

print("Dimensiones del bloque útil:", df_block.shape)
display(df_block.head(5))
display(df_block.tail(5))

Dimensiones del bloque útil: (1640, 4)


,raw_label,both_sexes,women,men
7,Total Nacional,NaN,NaN,NaN
8,Todas las edades,NaN,NaN,NaN
9,2023,28049.94,25591.31,30372.49
10,2022,26948.87,24359.82,29381.84
11,2021,25896.82,23175.95,28388.69


,raw_label,both_sexes,women,men
1642,2012,23494.49,-20597.19,-25361.7
1643,2011,25500.59,-21576,-28084.19
1644,2010,-25266.98,-19736.54,-28756.01
1645,NaN,NaN,NaN,NaN
1646,NaN,NaN,NaN,NaN


In [145]:
# ============================================================
# 7. LIMPIAR FILAS VACÍAS DEL BLOQUE
# ============================================================

# Convertimos cadenas vacías en NaN
df_block["raw_label"] = df_block["raw_label"].replace(r"^\s*$", np.nan, regex=True)

# Quitamos filas con raw_label vacío
df_block = df_block[df_block["raw_label"].notna()].copy()

# Pasamos a string y limpiamos espacios
df_block["raw_label"] = df_block["raw_label"].astype(str).str.strip()

# Quitamos basura que a veces se convierte en texto
df_block = df_block[~df_block["raw_label"].str.lower().isin(["nan", "none", ""])] .copy()

# Reseteamos índice para trabajar cómodos
df_block = df_block.reset_index(drop=True)

print("Dimensiones tras limpiar filas vacías:", df_block.shape)
display(df_block.head(5))
display(df_block.tail(5))

Dimensiones tras limpiar filas vacías: (1638, 4)


,raw_label,both_sexes,women,men
0,Total Nacional,NaN,NaN,NaN
1,Todas las edades,NaN,NaN,NaN
2,2023,28049.94,25591.31,30372.49
3,2022,26948.87,24359.82,29381.84
4,2021,25896.82,23175.95,28388.69


,raw_label,both_sexes,women,men
1633,2014,25604.47,-20739.28,-28897.03
1634,2013,23332.74,-21816.87,24350.32
1635,2012,23494.49,-20597.19,-25361.7
1636,2011,25500.59,-21576,-28084.19
1637,2010,-25266.98,-19736.54,-28756.01


In [146]:
# ============================================================
# 8. DEFINIR LOS GRUPOS DE EDAD
  # Esto nos ayuda a distinguir filas de edad frente a territorio.
# ============================================================

age_groups_ine = {
    "Todas las edades",
    "Menos de 25 años",
    "De 25 a 34 años",
    "De 35 a 44 años",
    "De 45 a 54 años",
    "55 y más años",
}

print("Grupos de edad detectables:")
for age in sorted(age_groups_ine):
    print("-", age)

Grupos de edad detectables:
- 55 y más años
- De 25 a 34 años
- De 35 a 44 años
- De 45 a 54 años
- Menos de 25 años
- Todas las edades


In [147]:
# ============================================================
# 9. FUNCIÓN AUXILIAR PARA DETECTAR AÑOS
  # Un año válido será un entero entre 2010 y 2023.
# ============================================================

def is_valid_year(value):
    """
    Devuelve True si el valor representa un año entre 2010 y 2023.
    """
    try:
        year = int(float(value))
        return 2010 <= year <= 2023
    except:
        return False

# Comprobaciones rápidas
print(is_valid_year("2023"))            # True
print(is_valid_year("Total Nacional"))  # False
print(is_valid_year("55 y más años"))   # False

True
False
False


In [148]:
# ============================================================
# 10. RECONSTRUIR LA JERARQUÍA Y PASAR A TABLA LARGA
  # Recorremos fila a fila:
    # - si la fila es un territorio, actualizamos territory
    # - si es un grupo de edad, actualizamos age_group
    # - si es un año, generamos 3 filas:
    #   Total, Mujeres, Hombres
# ============================================================

records = []

current_territory = None
current_age_group = None

for _, row in df_block.iterrows():
    label = row["raw_label"]

    # Caso 1: grupo de edad
    if label in age_groups_ine:
        current_age_group = label
        continue

    # Caso 2: año
    if is_valid_year(label):
        year = int(float(label))

        if current_territory is not None and current_age_group is not None:
            records.append({
                "territory_name": current_territory,
                "age_group": current_age_group,
                "year": year,
                "sex": "Ambos",
                "salary_raw": row["both_sexes"]
            })
            records.append({
                "territory_name": current_territory,
                "age_group": current_age_group,
                "year": year,
                "sex": "Mujeres",
                "salary_raw": row["women"]
            })
            records.append({
                "territory_name": current_territory,
                "age_group": current_age_group,
                "year": year,
                "sex": "Hombres",
                "salary_raw": row["men"]
            })
        continue

    # Caso 3: territorio
    current_territory = label
    current_age_group = None

# Convertimos a DataFrame
df_long = pd.DataFrame(records)

print("Dimensiones de la tabla larga:", df_long.shape)
display(df_long.head(20))
display(df_long.tail(20))

Dimensiones de la tabla larga: (4536, 5)


,territory_name,age_group,year,sex,salary_raw
0,Total Nacional,Todas las edades,2023,Ambos,28049.94
1,Total Nacional,Todas las edades,2023,Mujeres,25591.31
2,Total Nacional,Todas las edades,2023,Hombres,30372.49
3,Total Nacional,Todas las edades,2022,Ambos,26948.87
4,Total Nacional,Todas las edades,2022,Mujeres,24359.82
5,Total Nacional,Todas las edades,2022,Hombres,29381.84
6,Total Nacional,Todas las edades,2021,Ambos,25896.82
7,Total Nacional,Todas las edades,2021,Mujeres,23175.95
8,Total Nacional,Todas las edades,2021,Hombres,28388.69
9,Total Nacional,Todas las edades,2020,Ambos,25165.51


,territory_name,age_group,year,sex,salary_raw
4516,"Rioja, La",55 y más años,2016,Mujeres,-20880.48
4517,"Rioja, La",55 y más años,2016,Hombres,27056.47
4518,"Rioja, La",55 y más años,2015,Ambos,24730.65
4519,"Rioja, La",55 y más años,2015,Mujeres,-20850.75
4520,"Rioja, La",55 y más años,2015,Hombres,27642.02
4521,"Rioja, La",55 y más años,2014,Ambos,25604.47
4522,"Rioja, La",55 y más años,2014,Mujeres,-20739.28
4523,"Rioja, La",55 y más años,2014,Hombres,-28897.03
4524,"Rioja, La",55 y más años,2013,Ambos,23332.74
4525,"Rioja, La",55 y más años,2013,Mujeres,-21816.87


In [149]:
# ============================================================
# 11. COMPROBACIONES BÁSICAS DE COHERENCIA
# ============================================================

print("Número de filas:", len(df_long))
print("Territorios únicos:", df_long["territory_name"].nunique())
print("Grupos de edad únicos:", sorted(df_long["age_group"].unique()))
print("Sexos únicos:", sorted(df_long["sex"].unique()))
print("Año mínimo:", df_long["year"].min())
print("Año máximo:", df_long["year"].max())

# Comprobación de tamaño esperado
expected_rows = 18 * 6 * 14 * 3
print("Filas esperadas:", expected_rows)

Número de filas: 4536
Territorios únicos: 18
Grupos de edad únicos: ['55 y más años', 'De 25 a 34 años', 'De 35 a 44 años', 'De 45 a 54 años', 'Menos de 25 años', 'Todas las edades']
Sexos únicos: ['Ambos', 'Hombres', 'Mujeres']
Año mínimo: 2010
Año máximo: 2023
Filas esperadas: 4536


In [150]:
# ============================================================
# 12. LIMPIAR VALORES ESPECIALES
  # Reglas del INE:
    # - '..'  => dato no facilitado por muestra < 100 -> faltante
    # - signo negativo delante => dato con muestra entre 100 y 500
    #   y alta variabilidad. NO es salario negativo real.
# ============================================================

def parse_salary_value(value):
    """
    Devuelve:
    - salary_annual_eur
    - salary_missing_flag
    - salary_low_sample_flag
    """
    # Caso 1: faltante real
    if pd.isna(value):
        return np.nan, 1, 0

    value_str = str(value).strip()

    if value_str == "..":
        return np.nan, 1, 0

    # Intentamos convertir a número
    try:
        numeric_value = float(value_str)

        # Si es negativo, tomamos valor absoluto y marcamos flag
        if numeric_value < 0:
            return abs(numeric_value), 0, 1

        return numeric_value, 0, 0

    except:
        return np.nan, 1, 0


# Aplicamos la función a salary_raw
parsed_values = df_long["salary_raw"].apply(parse_salary_value)

df_long["salary_annual_eur"] = parsed_values.apply(lambda x: x[0])
df_long["salary_missing_flag"] = parsed_values.apply(lambda x: x[1])
df_long["salary_low_sample_flag"] = parsed_values.apply(lambda x: x[2])

display(df_long.head(70))

,territory_name,age_group,year,sex,salary_raw,salary_annual_eur,salary_missing_flag,salary_low_sample_flag
0,Total Nacional,Todas las edades,2023,Ambos,28049.94,28049.94,0,0
1,Total Nacional,Todas las edades,2023,Mujeres,25591.31,25591.31,0,0
2,Total Nacional,Todas las edades,2023,Hombres,30372.49,30372.49,0,0
3,Total Nacional,Todas las edades,2022,Ambos,26948.87,26948.87,0,0
4,Total Nacional,Todas las edades,2022,Mujeres,24359.82,24359.82,0,0
...,...,...,...,...,...,...,...,...
65,Total Nacional,Menos de 25 años,2016,Hombres,12358.6,12358.60,0,0
66,Total Nacional,Menos de 25 años,2015,Ambos,11039.53,11039.53,0,0
67,Total Nacional,Menos de 25 años,2015,Mujeres,9534.67,9534.67,0,0
68,Total Nacional,Menos de 25 años,2015,Hombres,12454.32,12454.32,0,0


In [151]:
# ============================================================
#   COMPROBACIONES:
# ============================================================

In [152]:
# ============================================================
# TEST_01. VER EJEMPLOS DE LOS TRES CASOS
# - valor normal
# - valor faltante
# - valor con baja muestra (negativo en la fuente)
# ============================================================

# Valores faltantes
display(
    df_long[df_long["salary_missing_flag"] == 1][
        ["territory_name", "age_group", "year", "sex", "salary_raw", "salary_annual_eur", "salary_missing_flag", "salary_low_sample_flag"]
    ].head(5)
)

# Valores de baja muestra
display(
    df_long[df_long["salary_low_sample_flag"] == 1][
        ["territory_name", "age_group", "year", "sex", "salary_raw", "salary_annual_eur", "salary_missing_flag", "salary_low_sample_flag"]
    ].head(5)
)

# Valores normales
display(
    df_long[
        (df_long["salary_missing_flag"] == 0) &
        (df_long["salary_low_sample_flag"] == 0)
    ][
        ["territory_name", "age_group", "year", "sex", "salary_raw", "salary_annual_eur", "salary_missing_flag", "salary_low_sample_flag"]
    ].head(5)
)

,territory_name,age_group,year,sex,salary_raw,salary_annual_eur,salary_missing_flag,salary_low_sample_flag
553,Aragón,Menos de 25 años,2021,Mujeres,..,NaN,1,0
556,Aragón,Menos de 25 años,2020,Mujeres,..,NaN,1,0
559,Aragón,Menos de 25 años,2019,Mujeres,..,NaN,1,0
565,Aragón,Menos de 25 años,2017,Mujeres,..,NaN,1,0
568,Aragón,Menos de 25 años,2016,Mujeres,..,NaN,1,0


,territory_name,age_group,year,sex,salary_raw,salary_annual_eur,salary_missing_flag,salary_low_sample_flag
295,Andalucía,Menos de 25 años,2023,Mujeres,-12785.1,12785.10,0,1
298,Andalucía,Menos de 25 años,2022,Mujeres,-12055.16,12055.16,0,1
299,Andalucía,Menos de 25 años,2022,Hombres,-16443.75,16443.75,0,1
301,Andalucía,Menos de 25 años,2021,Mujeres,-11646.17,11646.17,0,1
302,Andalucía,Menos de 25 años,2021,Hombres,-13389.36,13389.36,0,1


,territory_name,age_group,year,sex,salary_raw,salary_annual_eur,salary_missing_flag,salary_low_sample_flag
0,Total Nacional,Todas las edades,2023,Ambos,28049.94,28049.94,0,0
1,Total Nacional,Todas las edades,2023,Mujeres,25591.31,25591.31,0,0
2,Total Nacional,Todas las edades,2023,Hombres,30372.49,30372.49,0,0
3,Total Nacional,Todas las edades,2022,Ambos,26948.87,26948.87,0,0
4,Total Nacional,Todas las edades,2022,Mujeres,24359.82,24359.82,0,0


In [153]:
# ============================================================
# TEST_02. ASEGURARSE DE QUE NO QUEDAN SALARIOS NEGATIVOS
# ============================================================

negative_salaries = df_long[df_long["salary_annual_eur"] < 0]

print("Número de salarios negativos después de limpiar:", len(negative_salaries))
display(negative_salaries.head())

Número de salarios negativos después de limpiar: 0


,territory_name,age_group,year,sex,salary_raw,salary_annual_eur,salary_missing_flag,salary_low_sample_flag


In [154]:
# ============================================================
# TEST_03. COMPROBAR QUE LOS NEGATIVOS ORIGINALES AHORA
# SON POSITIVOS Y ESTÁN MARCADOS COMO BAJA MUESTRA
# ============================================================

# Filas donde el valor bruto era negativo
neg_raw = df_long[pd.to_numeric(df_long["salary_raw"], errors="coerce") < 0].copy()

print("Número de valores brutos negativos:", len(neg_raw))

display(
    neg_raw[
        ["territory_name", "age_group", "year", "sex", "salary_raw", "salary_annual_eur", "salary_missing_flag", "salary_low_sample_flag"]
    ].head(15)
)

Número de valores brutos negativos: 607


,territory_name,age_group,year,sex,salary_raw,salary_annual_eur,salary_missing_flag,salary_low_sample_flag
295,Andalucía,Menos de 25 años,2023,Mujeres,-12785.1,12785.10,0,1
298,Andalucía,Menos de 25 años,2022,Mujeres,-12055.16,12055.16,0,1
299,Andalucía,Menos de 25 años,2022,Hombres,-16443.75,16443.75,0,1
301,Andalucía,Menos de 25 años,2021,Mujeres,-11646.17,11646.17,0,1
302,Andalucía,Menos de 25 años,2021,Hombres,-13389.36,13389.36,0,1
303,Andalucía,Menos de 25 años,2020,Ambos,-11313.88,11313.88,0,1
304,Andalucía,Menos de 25 años,2020,Mujeres,-10674.17,10674.17,0,1
305,Andalucía,Menos de 25 años,2020,Hombres,-11920.68,11920.68,0,1
307,Andalucía,Menos de 25 años,2019,Mujeres,-10615.13,10615.13,0,1
308,Andalucía,Menos de 25 años,2019,Hombres,-12772.33,12772.33,0,1


In [155]:
# ============================================================
# TEST_04. COMPROBAR QUE LOS '..' SE HAN CONVERTIDO EN NULO
# ============================================================

dots_rows = df_long[df_long["salary_raw"].astype(str).str.strip() == ".."].copy()

print("Número de filas con '..' en salary_raw:", len(dots_rows))

display(
    dots_rows[
        ["territory_name", "age_group", "year", "sex", "salary_raw", "salary_annual_eur", "salary_missing_flag"]
    ].head(15)
)

Número de filas con '..' en salary_raw: 163


,territory_name,age_group,year,sex,salary_raw,salary_annual_eur,salary_missing_flag
553,Aragón,Menos de 25 años,2021,Mujeres,..,NaN,1
556,Aragón,Menos de 25 años,2020,Mujeres,..,NaN,1
559,Aragón,Menos de 25 años,2019,Mujeres,..,NaN,1
565,Aragón,Menos de 25 años,2017,Mujeres,..,NaN,1
568,Aragón,Menos de 25 años,2016,Mujeres,..,NaN,1
569,Aragón,Menos de 25 años,2016,Hombres,..,NaN,1
571,Aragón,Menos de 25 años,2015,Mujeres,..,NaN,1
577,Aragón,Menos de 25 años,2013,Mujeres,..,NaN,1
580,Aragón,Menos de 25 años,2012,Mujeres,..,NaN,1
799,"Asturias, Principado de",Menos de 25 años,2023,Mujeres,..,NaN,1


In [156]:
# ============================================================
# TEST_05. RESUMEN GENERAL DE CALIDAD
# ============================================================

print("Total filas:", len(df_long))
print("Nulos en salary_annual_eur:", df_long["salary_annual_eur"].isna().sum())
print("Missing flag = 1:", df_long["salary_missing_flag"].sum())
print("Low sample flag = 1:", df_long["salary_low_sample_flag"].sum())
print("Salarios negativos tras limpieza:", (df_long["salary_annual_eur"] < 0).sum())

Total filas: 4536
Nulos en salary_annual_eur: 163
Missing flag = 1: 163
Low sample flag = 1: 607
Salarios negativos tras limpieza: 0


In [157]:
# ============================================================
# TEST_06. COMPROBACIÓN MANUAL DE UNA FILA CONCRETA
# Cambia los valores por un caso que hayas visto en el Excel.
# ============================================================

example = df_long[
    (df_long["territory_name"] == "Aragón") &
    (df_long["age_group"] == "Menos de 25 años") &
    (df_long["year"] == 2012) &
    (df_long["sex"] == "Ambos")
]

display(example)

,territory_name,age_group,year,sex,salary_raw,salary_annual_eur,salary_missing_flag,salary_low_sample_flag
579,Aragón,Menos de 25 años,2012,Ambos,-11156.57,11156.57,0,1


In [158]:
# ============================================================
# 13. NORMALIZAR NOMBRES DE TERRITORIOS
  # Queremos dejar los nombres consistentes para poder cruzarlos
    # luego con alquiler, paro y población.
# ============================================================

territory_map = {
    "Total Nacional": "España",
    "Andalucía": "Andalucía",
    "Aragón": "Aragón",
    "Asturias, Principado de": "Asturias",
    "Balears, Illes": "Baleares",
    "Canarias": "Canarias",
    "Cantabria": "Cantabria",
    "Castilla y León": "Castilla y León",
    "Castilla - La Mancha": "Castilla-La Mancha",
    "Cataluña": "Cataluña",
    "Comunitat Valenciana": "Comunidad Valenciana",
    "Extremadura": "Extremadura",
    "Galicia": "Galicia",
    "Madrid, Comunidad de": "Madrid",
    "Murcia, Región de": "Murcia",
    "Navarra, Comunidad Foral de": "Navarra",
    "País Vasco": "País Vasco",
    "Rioja, La": "La Rioja",
}

df_long["territory_name"] = df_long["territory_name"].replace(territory_map)

print("Territorios normalizados:")
print(sorted(df_long["territory_name"].unique()))
print("\nNúmero de territorios:", df_long["territory_name"].nunique())

Territorios normalizados:
['Andalucía', 'Aragón', 'Asturias', 'Baleares', 'Canarias', 'Cantabria', 'Castilla y León', 'Castilla-La Mancha', 'Cataluña', 'Comunidad Valenciana', 'España', 'Extremadura', 'Galicia', 'La Rioja', 'Madrid', 'Murcia', 'Navarra', 'País Vasco']

Número de territorios: 18


In [159]:
# Distinguimos España del resto de comunidades autónomas.

df_long["territory_type"] = np.where(
    df_long["territory_name"] == "España",
    "España",
    "CCAA"
)

display(
    df_long[["territory_name", "territory_type"]]
    .drop_duplicates()
    .sort_values(["territory_type", "territory_name"])
)

,territory_name,territory_type
252,Andalucía,CCAA
504,Aragón,CCAA
756,Asturias,CCAA
1008,Baleares,CCAA
1260,Canarias,CCAA
1512,Cantabria,CCAA
1764,Castilla y León,CCAA
2016,Castilla-La Mancha,CCAA
2268,Cataluña,CCAA
2520,Comunidad Valenciana,CCAA


In [160]:
# ============================================================
# 14. NORMALIZAR GRUPOS DE EDAD
# ============================================================

age_group_map = {
    "Todas las edades": "all_ages",
    "Menos de 25 años": "lt_25",
    "De 25 a 34 años": "25_34",
    "De 35 a 44 años": "35_44",
    "De 45 a 54 años": "45_54",
    "55 y más años": "55_plus",
}

df_long["age_group"] = df_long["age_group"].replace(age_group_map)

print("Grupos de edad normalizados:")
print(sorted(df_long["age_group"].unique()))

Grupos de edad normalizados:
['25_34', '35_44', '45_54', '55_plus', 'all_ages', 'lt_25']


In [161]:
# ============================================================
# 16. CONSTRUIR LA TABLA FINAL DE SALARIOS
  # Dejamos ya solo las columnas definitivas.
# ============================================================

df_salary = df_long.copy()

df_salary["salary_source"] = "INE_EES_28201"

df_salary = df_salary[
    [
        "territory_name",
        "territory_type",
        "year",
        "sex",
        "age_group",
        "salary_annual_eur",
        "salary_missing_flag",
        "salary_low_sample_flag",
        "salary_source",
    ]
].copy()

print("Dimensiones de la tabla final de salarios:", df_salary.shape)
display(df_salary.head(5))
display(df_salary.tail(5))

Dimensiones de la tabla final de salarios: (4536, 9)


,territory_name,territory_type,year,sex,age_group,salary_annual_eur,salary_missing_flag,salary_low_sample_flag,salary_source
0,España,España,2023,Ambos,all_ages,28049.94,0,0,INE_EES_28201
1,España,España,2023,Mujeres,all_ages,25591.31,0,0,INE_EES_28201
2,España,España,2023,Hombres,all_ages,30372.49,0,0,INE_EES_28201
3,España,España,2022,Ambos,all_ages,26948.87,0,0,INE_EES_28201
4,España,España,2022,Mujeres,all_ages,24359.82,0,0,INE_EES_28201


,territory_name,territory_type,year,sex,age_group,salary_annual_eur,salary_missing_flag,salary_low_sample_flag,salary_source
4531,La Rioja,CCAA,2011,Mujeres,55_plus,21576.00,0,1,INE_EES_28201
4532,La Rioja,CCAA,2011,Hombres,55_plus,28084.19,0,1,INE_EES_28201
4533,La Rioja,CCAA,2010,Ambos,55_plus,25266.98,0,1,INE_EES_28201
4534,La Rioja,CCAA,2010,Mujeres,55_plus,19736.54,0,1,INE_EES_28201
4535,La Rioja,CCAA,2010,Hombres,55_plus,28756.01,0,1,INE_EES_28201


In [162]:
# ============================================================
# 17. ORDENAR Y COMPROBAR LA TABLA FINAL
# ============================================================

df_salary = df_salary.sort_values(
    by=["territory_type", "territory_name", "age_group", "year", "sex"]
).reset_index(drop=True)

print("Número de filas:", len(df_salary))
print("Territorios únicos:", df_salary["territory_name"].nunique())
print("Sexos únicos:", sorted(df_salary["sex"].unique()))
print("Grupos de edad:", sorted(df_salary["age_group"].unique()))
print("Rango temporal:", df_salary["year"].min(), "-", df_salary["year"].max())
print("Nulos salariales:", int(df_salary["salary_missing_flag"].sum()))
print("Datos con baja muestra:", int(df_salary["salary_low_sample_flag"].sum()))

display(df_salary.head(20))

Número de filas: 4536
Territorios únicos: 18
Sexos únicos: ['Ambos', 'Hombres', 'Mujeres']
Grupos de edad: ['25_34', '35_44', '45_54', '55_plus', 'all_ages', 'lt_25']
Rango temporal: 2010 - 2023
Nulos salariales: 163
Datos con baja muestra: 607


,territory_name,territory_type,year,sex,age_group,salary_annual_eur,salary_missing_flag,salary_low_sample_flag,salary_source
0,Andalucía,CCAA,2010,Ambos,25_34,17299.79,0,0,INE_EES_28201
1,Andalucía,CCAA,2010,Hombres,25_34,19574.29,0,0,INE_EES_28201
2,Andalucía,CCAA,2010,Mujeres,25_34,15130.18,0,0,INE_EES_28201
3,Andalucía,CCAA,2011,Ambos,25_34,16969.88,0,0,INE_EES_28201
4,Andalucía,CCAA,2011,Hombres,25_34,18903.23,0,0,INE_EES_28201
5,Andalucía,CCAA,2011,Mujeres,25_34,15194.94,0,0,INE_EES_28201
6,Andalucía,CCAA,2012,Ambos,25_34,16476.61,0,0,INE_EES_28201
7,Andalucía,CCAA,2012,Hombres,25_34,18590.66,0,0,INE_EES_28201
8,Andalucía,CCAA,2012,Mujeres,25_34,14744.31,0,0,INE_EES_28201
9,Andalucía,CCAA,2013,Ambos,25_34,15480.24,0,0,INE_EES_28201


In [165]:
# ============================================================
# 18. GUARDAR EL CSV LIMPIO
# ============================================================

from pathlib import Path

# Creamos la carpeta de salida si no existe
Path(output_file).parent.mkdir(parents=True, exist_ok=True)

# Guardamos el CSV
df_salary.to_csv(output_file, index=False, encoding="utf-8-sig")

print("Archivo guardado en:")
print(output_file)

Archivo guardado en:
/content/drive/MyDrive/TFG-Accesibilidad-Vivienda-Joven/data/data clean/salary_observed.csv


In [166]:
# ============================================================
# 19. COMPROBAR CSV
# ============================================================

df_check = pd.read_csv(output_file)

print("Dimensiones del CSV guardado:", df_check.shape)
display(df_check.head(10))

Dimensiones del CSV guardado: (4536, 9)


,territory_name,territory_type,year,sex,age_group,salary_annual_eur,salary_missing_flag,salary_low_sample_flag,salary_source
0,Andalucía,CCAA,2010,Ambos,25_34,17299.79,0,0,INE_EES_28201
1,Andalucía,CCAA,2010,Hombres,25_34,19574.29,0,0,INE_EES_28201
2,Andalucía,CCAA,2010,Mujeres,25_34,15130.18,0,0,INE_EES_28201
3,Andalucía,CCAA,2011,Ambos,25_34,16969.88,0,0,INE_EES_28201
4,Andalucía,CCAA,2011,Hombres,25_34,18903.23,0,0,INE_EES_28201
5,Andalucía,CCAA,2011,Mujeres,25_34,15194.94,0,0,INE_EES_28201
6,Andalucía,CCAA,2012,Ambos,25_34,16476.61,0,0,INE_EES_28201
7,Andalucía,CCAA,2012,Hombres,25_34,18590.66,0,0,INE_EES_28201
8,Andalucía,CCAA,2012,Mujeres,25_34,14744.31,0,0,INE_EES_28201
9,Andalucía,CCAA,2013,Ambos,25_34,15480.24,0,0,INE_EES_28201


In [167]:
# ============================================================
# 20. INFORME DE CALIDAD DE LA FUENTE SALARIAL
# ============================================================

quality_report = {
    "n_rows": len(df_salary),
    "n_territories": df_salary["territory_name"].nunique(),
    "year_min": int(df_salary["year"].min()),
    "year_max": int(df_salary["year"].max()),
    "n_missing_salary": int(df_salary["salary_missing_flag"].sum()),
    "n_low_sample_salary": int(df_salary["salary_low_sample_flag"].sum()),
}

print("Resumen general:")
for k, v in quality_report.items():
    print(f"{k}: {v}")

print("\nNulos por grupo de edad:")
display(
    df_salary.groupby("age_group", as_index=False)["salary_missing_flag"].sum()
)

print("\nBaja muestra por grupo de edad:")
display(
    df_salary.groupby("age_group", as_index=False)["salary_low_sample_flag"].sum()
)

Resumen general:
n_rows: 4536
n_territories: 18
year_min: 2010
year_max: 2023
n_missing_salary: 163
n_low_sample_salary: 607

Nulos por grupo de edad:


,age_group,salary_missing_flag
0,25_34,0
1,35_44,0
2,45_54,0
3,55_plus,0
4,all_ages,0
5,lt_25,163



Baja muestra por grupo de edad:


,age_group,salary_low_sample_flag
0,25_34,48
1,35_44,0
2,45_54,4
3,55_plus,82
4,all_ages,0
5,lt_25,473
